# MNIST Digit Classification with Early Stopping

## Goal
Train a neural network to recognize handwritten digits from 0 to 9.
Use a custom callback to stop training when training accuracy reaches 98%.

In [21]:
import tensorflow as tf

## 1. Load the MNIST Data

MNIST contains 60,000 training images and 10,000 test images.
Each image is a 28 × 28 grayscale picture of a handwritten digit.
The labels are numbers from 0 to 9.

In [22]:
# Load train and test dataset
(train_image,train_label),(test_image,test_label)=tf.keras.datasets.mnist.load_data()

# Shape of image and label
print(f"Image shape: {train_image.shape}\nLabel shape: {train_label.shape} ")
print(f"There are {train_image.shape[0]} examples of image of shape ({train_image.shape[1]},{train_image.shape[2]})  ")



Image shape: (60000, 28, 28)
Label shape: (60000,) 
There are 60000 examples of image of shape (28,28)  


## 2. Normalize the Images

Pixel values originally range from 0 to 255. Dividing by 255 scales them
to the range 0 to 1, which helps the model learn.

In [23]:
# Normalize the image
train_image=train_image/255.0
test_image=test_image/255.0

## 3. Build and Compile the Model

`Flatten` turns each 28 × 28 image into 784 values. The hidden Dense layer
has 512 units and uses ReLU. The final layer has 10 units—one for each
digit—and uses softmax to produce a score for each digit.

The model uses Adam, sparse categorical cross-entropy loss, and tracks
accuracy.

In [24]:
# Create model and compile

def create_compile_model():
    model=tf.keras.models.Sequential([
        tf.keras.Input(shape=(28,28)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512,activation=tf.nn.relu),
        tf.keras.layers.Dense(10,activation=tf.nn.softmax)
    ])
    model.compile(optimizer=tf.optimizers.Adam(),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return model

## 4. Define the Early-Stopping Callback

At the end of each epoch, the callback checks the training accuracy.
When it reaches 98%, the callback sets `model.stop_training` to `True`
to end training.

In [ ]:
# Callback class define

class earlystop(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs = None):
        if logs['accuracy']>0.98:
            self.model.stop_training=True
            print("Reached 98% accuracy so cancelling training !")

## 5. Train the Model

Train the model for up to 10 epochs. The callback may stop training
earlier if the accuracy reaches the 98% target.

In [29]:
# Create model and fit 
def train_mnist(image, label):
    model = create_compile_model()
    history = model.fit(
        image, label,
        epochs=10,
        callbacks=[earlystop()],
        verbose=0
    )
    return model, history


## 6. Evaluate on the Test Data

Evaluate the trained model on the test images, which were not used
during training. This gives a separate measure of its performance.

In [30]:
model, training_history = train_mnist(train_image, train_label)
test_loss, test_accuracy = model.evaluate(test_image, test_label)
print(f"Test accuracy: {test_accuracy:.2%}")

Reached 985% accuracy so cancelling tranining !
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9814 - loss: 0.0653
Test accuracy: 98.14%


## Results

The test accuracy and test loss are shown above. Test accuracy measures
the proportion of test images classified correctly.